**Step 1:Installing Dependencies**

In [1]:
!pip install -q -U "transformers>=4.51.0" "huggingface_hub>=0.28.0" peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 84.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 35.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 89.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not i

**Step 2: Load Tokenizer & Dataset**

In [11]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 2. Load code Writing Dataset
ds = load_dataset("nickrosh/Evol-Instruct-Code-80k-v1", split="train")

print("--- DATASET SUMMARY ---")
print(f"Total raw rows: {len(ds)}")
print(f"Dataset columns: {ds.column_names}")

README.md:   0%|          | 0.00/282 [00:00<?, ?B/s]

EvolInstruct-Code-80k.json: reconstructing file:   0%|          |  0.00B /  121MB            

EvolInstruct-Code-80k.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78264 [00:00<?, ? examples/s]

--- DATASET SUMMARY ---
Total raw rows: 78264
Dataset columns: ['instruction', 'output']


**Step 3: Checking Token Length Distribution (1024 Context Audit)**

In [14]:
# Analyzes 1,000 samples using 'instruction' and 'output' columns for EvolInstruct

def calculate_tokens(example):
    instruction = str(example.get("instruction", "")).strip()
    output_text = str(example.get("output", "")).strip()

    messages = [
        {
            "role": "system", 
            "content": "You are an expert software engineer. Provide clean, correct, production-ready code with zero conversational preamble."
        },
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": output_text}
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False
    )
    
    tokens = tokenizer(formatted_text, truncation=False)["input_ids"]
    return {"token_count": len(tokens)}

# Sample 1,000 rows for fast auditing
sample_audit = ds.select(range(1000)).map(calculate_tokens)
lengths = sample_audit["token_count"]

valid_1024 = sum(1 for l in lengths if l <= 1024)
avg_len = sum(lengths) / len(lengths)

print("--- TOKEN LENGTH AUDIT (EVOL-INSTRUCT-CODE-80K) ---")
print(f"Average token length: {avg_len:.1f}")
print(f"Max token length in sample: {max(lengths)}")
print(f"Min token length in sample: {min(lengths)}")
print(f"Samples fitting within 1024 tokens: {valid_1024} / 1000 ({valid_1024/10:.1f}%)")

--- TOKEN LENGTH AUDIT (EVOL-INSTRUCT-CODE-80K) ---
Average token length: 575.0
Max token length in sample: 2004
Min token length in sample: 63
Samples fitting within 1024 tokens: 917 / 1000 (91.7%)


**Step 4: Visual Inspection of 5 Random Formatted Examples**

In [16]:
import random

random.seed(42)
sample_indices = random.sample(range(len(ds)), 5)

for i, idx in enumerate(sample_indices, 1):
    print(f"==================== EXAMPLE #{i} (Index: {idx}) ====================")
    instruction = str(ds[idx].get("instruction", "")).strip()
    output_text = str(ds[idx].get("output", "")).strip()

    messages = [
        {
            "role": "system", 
            "content": "You are an expert software engineer. Provide clean, correct, production-ready code with zero conversational preamble."
        },
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": output_text}
    ]

    formatted_sample = tokenizer.apply_chat_template(messages, tokenize=False)
    print(formatted_sample)
    print("=" * 65 + "\n")

==================== EXAMPLE #1 (Index: 14592) ====================
<|im_start|>system
You are an expert software engineer. Provide clean, correct, production-ready code with zero conversational preamble.<|im_end|>
<|im_start|>user
You are given a string "hello". Write a function to print out all the possible subsets of that string, but with the following additional constraints:
- The subsets should only include contiguous characters from the original string.
- The subsets should not contain any duplicate characters.
- The subsets should be printed in lexicographically sorted order.
- The function should return the subsets in reverse lexicographical order.
- The function should handle strings of length up to 100 characters.
- The function should have a time complexity of O(2^n), where n is the length of the string.

Rewritten Test:
You are given a string "hello". Write a function to print out all the possible subsets of that string, but with the following additional constraints:
- The 

**Step 5: Verify Response-Only Loss Masking (labels = -100)**

In [17]:
from trl import DataCollatorForCompletionOnlyLM

# 1. Setup Data Collator targeting Qwen assistant tokens
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)

# 2. Format a test sample from ds
sample_item = ds[0]
instruction = str(sample_item.get("instruction", "")).strip()
output_text = str(sample_item.get("output", "")).strip()

messages = [
    {
        "role": "system", 
        "content": "You are an expert software engineer. Provide clean, correct, production-ready code with zero conversational preamble."
    },
    {"role": "user", "content": instruction},
    {"role": "assistant", "content": output_text}
]

formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
tokenized = tokenizer(formatted_text, return_tensors="pt")

# 3. Pass through collator to verify labels masking
batch = collator([tokenized["input_ids"][0]])
labels = batch["labels"][0]

masked_tokens = sum(1 for label in labels if label == -100)
unmasked_tokens = sum(1 for label in labels if label != -100)

print("=== RESPONSE-ONLY LOSS MASKING VERIFICATION ===")
print(f"Total Sequence Length:  {len(labels)} tokens")
print(f"Masked Tokens (-100):   {masked_tokens} (System & User prompts ignored)")
print(f"Target Loss Tokens:     {unmasked_tokens} (Assistant output computed)")
print(f"Assistant Target Ratio: {(unmasked_tokens / len(labels)) * 100:.1f}%")

ModuleNotFoundError: No module named 'trl'

**Process Started**

**Step 1: Preprocessing & Filtering Cell**

In [ ]:
from datasets import load_dataset
import random

# 1. Load full dataset
raw_ds = load_dataset("euclaise/WritingPrompts_curated", split="train")

# 2. Map and tokenize with hard filtering
def preprocess_writing_data(example):
    prompt_text = str(example["prompt"]).strip()
    story_text = str(example["body"]).strip()

    prompt_messages = [{"role": "user", "content": prompt_text}]
    full_messages = [{"role": "user", "content": prompt_text}, {"role": "assistant", "content": story_text}]

    prompt_str = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    full_str = tokenizer.apply_chat_template(
        full_messages, tokenize=False, enable_thinking=False
    )

    prompt_ids = tokenizer(prompt_str, truncation=False)["input_ids"]
    full_ids = tokenizer(full_str, truncation=False)["input_ids"]

    # Calculate token length
    total_len = len(full_ids)
    
    # Return empty if over context budget (filtered out in next step)
    if total_len > 1024 or total_len < 100:
        return {"input_ids": [], "labels": [], "attention_mask": [], "valid": False}

    prompt_len = len(prompt_ids)
    labels = [-100] * prompt_len + full_ids[prompt_len:]

    return {
        "input_ids": full_ids,
        "labels": labels,
        "attention_mask": [1] * total_len,
        "valid": True
    }

print("Preprocessing and filtering dataset...")
processed_ds = raw_ds.map(preprocess_writing_data, remove_columns=raw_ds.column_names)

# Filter out invalid/oversized rows
clean_ds = processed_ds.filter(lambda x: x["valid"]).remove_columns(["valid"])

# Shuffle and select 15,000 rows
train_ds = clean_ds.shuffle(seed=42).select(range(min(15000, len(clean_ds))))

print(f"Filtered Dataset Ready! Training samples: {len(train_ds)}")

**Step 2: Training Execution Cell (Creative LoRA)**

In [ ]:
import os
import shutil
import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
OUTPUT_DIR = "./creative_lora_qwen3"
FINAL_DIR = "./final_creative_lora"

# 1. Quantization Config (NF4 4-bit with Double Quantization enabled)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 2. Load Base Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 3. Model Optimizations for QLoRA Training
model.gradient_checkpointing_enable()
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# 4. Apply LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 5. Training Arguments (Cleaned for Kaggle Environment)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    optim="paged_adamw_8bit",          # Prevents VRAM spikes during backward passes
    gradient_checkpointing=True,       # Saves activation memory
    lr_scheduler_type="cosine",        # Cosine decay schedule
    warmup_steps=18,                   # Ramps up LR over first 18 steps
    logging_steps=10,
    max_steps=600,                     # ~1.5 hours on T4
    fp16=True,                         # Turing GPU support
    bf16=False,
    save_strategy="steps",
    save_steps=150,
    save_total_limit=2,                # Keeps disk usage under control
    seed=42,
    report_to="none"
)

# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    train_dataset=train_ds,
    args=training_args,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer, 
        pad_to_multiple_of=8, 
        return_tensors="pt"
    )
)

# 7. Checkpoint Resume Logic & Execution
last_checkpoint = None
if os.path.exists(OUTPUT_DIR):
    checkpoints = [
        os.path.join(OUTPUT_DIR, d) 
        for d in os.listdir(OUTPUT_DIR) 
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        last_checkpoint = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]

if last_checkpoint:
    print(f"Resuming training from checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting fresh Creative LoRA Training...")
    trainer.train()

# 8. Save Weights and Package for Kaggle Persistence
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
shutil.make_archive("final_creative_lora", 'zip', FINAL_DIR)

print(
    f"\nCreative LoRA training complete!\n"
    f"1. Model saved locally to: {FINAL_DIR}\n"
    f"2. Packaged zip created: final_creative_lora.zip (Downloadable from Kaggle Output directory)"
)

**Testing**

In [ ]:
import torch

# 1. CRITICAL: Re-enable KV Cache for generation and set Evaluation Mode
model.config.use_cache = True
model.eval()

# 2. Fix pad token fallback
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

test_prompts = [
    "You are an ancient lighthouse keeper who discovers the light isn't warning ships away from the rocks—it's attracting something in the ocean.",
    "A detective receives a case to investigate their own disappearance."
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n==================== TEST PROMPT #{i} ====================")
    print(f"PROMPT: {prompt}\n")

    messages = [{"role": "user", "content": prompt}]
    formatted_input = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(formatted_input, return_tensors="pt").to("cuda")

    # --- FINE-TUNED CREATIVE ADAPTER ---
    print("--- FINE-TUNED CREATIVE ADAPTER ---")
    with torch.no_grad():
        lora_outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.1
        )
    print(tokenizer.decode(lora_outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

    # --- BASE QWEN BASELINE ---
    print("\n--- BASE QWEN3-4B BASELINE ---")
    with model.disable_adapter():
        with torch.no_grad():
            base_outputs = model.generate(
                **inputs,
                max_new_tokens=250,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                repetition_penalty=1.1
            )
    print(tokenizer.decode(base_outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))
    print("=" * 60)

In [ ]:
import math
import torch

# Ensure model and tokenizer settings for inference
model.config.use_cache = True
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

test_dataset_prompts = [
    "A world where human souls physically manifest as weapons upon turning eighteen.",
    "You are an immortal librarian in a city that forgets its history every hundred years.",
    "An AI designed to predict crime predicts that its creator will commit a murder in one hour.",
    "You buy an old watch at a flea market, only to find it ticks backward and rewinds local time.",
    "A space station captain receives a distress call originating from inside their own ship."
]

def calculate_distinct_n(texts, n=2):
    tokens = [t.lower() for text in texts for t in text.split()]
    if len(tokens) < n:
        return 0.0
    ngrams = set(zip(*[tokens[i:] for i in range(n)]))
    return len(ngrams) / (len(tokens) - n + 1)

def evaluate_perplexity_and_diversity(use_adapter=True):
    generated_texts = []
    total_loss = 0.0
    
    for prompt in test_dataset_prompts:
        formatted = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}], 
            tokenize=False, 
            add_generation_prompt=True
        )
        inputs = tokenizer(formatted, return_tensors="pt").to("cuda")
        
        # Toggle adapter using context manager for base model vs adapter
        with torch.no_grad():
            if not use_adapter:
                with model.disable_adapter():
                    outputs = model.generate(
                        **inputs, max_new_tokens=200, do_sample=True, temperature=0.7, 
                        top_p=0.9, pad_token_id=tokenizer.pad_token_id, repetition_penalty=1.1
                    )
                    loss = model(**inputs, labels=inputs["input_ids"].clone()).loss
            else:
                outputs = model.generate(
                    **inputs, max_new_tokens=200, do_sample=True, temperature=0.7, 
                    top_p=0.9, pad_token_id=tokenizer.pad_token_id, repetition_penalty=1.1
                )
                loss = model(**inputs, labels=inputs["input_ids"].clone()).loss

            text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            generated_texts.append(text)
            total_loss += loss.item()

    avg_loss = total_loss / len(test_dataset_prompts)
    ppl = math.exp(avg_loss)
    distinct_1 = calculate_distinct_n(generated_texts, n=1)
    distinct_2 = calculate_distinct_n(generated_texts, n=2)

    return {"Loss": round(avg_loss, 4), "Perplexity": round(ppl, 2), "Distinct-1": round(distinct_1, 4), "Distinct-2": round(distinct_2, 4)}, generated_texts

print("Benchmarking Fine-Tuned Adapter...")
adapter_metrics, adapter_samples = evaluate_perplexity_and_diversity(use_adapter=True)

print("Benchmarking Base Model Baseline...")
base_metrics, base_samples = evaluate_perplexity_and_diversity(use_adapter=False)

print("\n=== QUANTITATIVE BENCHMARK RESULTS ===")
print(f"Fine-Tuned Adapter : {adapter_metrics}")
print(f"Base Model Baseline: {base_metrics}")

In [ ]:
# Format and print the actual generated stories for your LLM Judge
for i, (prompt, adapter_text, base_text) in enumerate(zip(test_dataset_prompts, adapter_samples, base_samples), 1):
    print(f"==================== PROMPT #{i} ====================")
    print(f"PROMPT: {prompt}\n")
    
    print("--- MODEL A (Fine-Tuned Adapter) ---")
    print(adapter_text.strip())
    print("\n--- MODEL B (Base Model) ---")
    print(base_text.strip())
    print("\n" + "="*60 + "\n")